# 03 · Qwen translation baseline

Baseline: give Qwen a Chinese post, get English back. No slang definitions yet.

Before running (Colab):
1. Change runtime to A100.
2. Upload data/processed/usual_splits.csv (I added this to the code because I kept forgetting to add it)


## 1. Install and import

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from google.colab import files

uploaded = files.upload() # usual_splits.csv
print(list(uploaded.keys())) # should print ['usual_splits.csv'] to make sure it uploaded the right thing

Saving usual_splits.csv to usual_splits.csv
['usual_splits.csv']


In [2]:
# src for using a HF model (plus Assignment 3): https://medium.com/data-science/choosing-and-implementing-hugging-face-models-026d71426fbe$0
%pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 144.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [2]:
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [5]:
# Check that I'm using the right GPU
print(torch.cuda.get_device_name(0))

NVIDIA L4


## 2. Load the model

In [6]:
#model: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct$0
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Structure of this code is based on Assignment 3 & the model's Quickstart guide
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model = AutoModelForCausalLM.from_pretrained( #parameters from the Quickstart documentation
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

#make sure it loaded on the right device
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded on: cuda:0


## 3. Helper function: translate one post

Since Qwen is a chat model, we need to send it a system and user message:
- system message saying what its job is
- user message containing the post

In [7]:
# Based on the Quickstart guide here: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct$0
system_prompt = (
    "You are a professional translator. "
    "Translate the user's Chinese social media post into natural English. "
    "Output only the English translation, with no notes or explanations."
)


def translate(chinese_text):
    # build the conversation
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": chinese_text},
    ]

    # Turn into Qwen's expected text format
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True, #from Quickstart guide
    )

    # turn the text into tokens & move to the same device as the model (so it can run quickly)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # generate the answer
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False, #always pick the most likely next token so the same post always gets the same translation
    )

    # the output contains the prompt AND the answer, so keep only the new tokens (the answer), then turn them back into text.
    prompt_length = inputs["input_ids"].shape[1]
    answer_tokens = output_tokens[0][prompt_length:]
    english_text = tokenizer.decode(answer_tokens, skip_special_tokens=True) #only decode the answer tokens (not the prompt)

    return english_text.strip()

## 4. Try it on one post

Test out the prompt/messages to make sure it just produced a translation without any notes.

In [8]:
example = "气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车"

start = time.time()
result = translate(example)
seconds = time.time() - start

print("Chinese:", example)
print("English:", result)
print("Took", round(seconds, 1), "seconds")

Chinese: 气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车
English: My sister is so frustrated. Is Fast Second almost out of service? F*ck, over an hour and a half and still no vehicle has arrived.
Took 3.2 seconds


Look at 10 slang and 10 non-slang posts from the test split to make sure everything looks right (no Chinese, no notes, etc.). Time it to get an estimate of the full runtime.

In [15]:
data = pd.read_csv("usual_splits.csv")
test_posts = data[data["split"] == "test"]

# get 10 random test posts with and without slang
slang_sample = test_posts[test_posts["has_slang_simple"] == True].sample(10, random_state=266) #use random state so it's repeatable
no_slang_sample = test_posts[test_posts["has_slang_simple"] == False].sample(10, random_state=266)

# Stack the two samples into one table
sample = pd.concat([slang_sample, no_slang_sample])


In [16]:
translations = []

#time it to give me an idea of what we're working with
start = time.time()

for text in sample["text"]:
    english = translate(text)
    translations.append(english)
    print(".")      # one dot per post to track progress

seconds = time.time() - start #get amount of time it took to run
print()
print(f"Num posts: {len(translations)}")
print("Total time: {round(seconds)} s")
print(f"Est. time per post: {round(seconds/len(translations),1)} s")

sample["translation"] = translations

.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

Done: 20 posts in 46 seconds
About 2.3 seconds per post


In [18]:
# Look at the results side by side
for index, row in sample.iterrows():
    print("[" + row["label"] + "]", "slang" if row["has_slang_simple"] else "no slang")
    print("  ZH:", row["text"])
    print("  EN:", row["translation"])
    print()

[sad] slang
  ZH: #周艺轩##UNIQ周艺轩#可怜的孩子在家跟一大堆作业一起过国庆[怒][失望][抓狂][打脸][吐][生病][拜拜][晕][困][伤心][惊恐][惊讶][害怕][心碎]
  EN: #Zhou Yixuan# #UNIQ Zhou Yixuan# Poor kid had to do homework during the National Day holiday [Angry][Disappointed][Annoyed][Embarrassed][Nauseated][Sick][Bye][Dizzy][Tired][Heartbroken][Surprised][Scared][Heartbroken]

[angry] slang
  ZH: 来来[勾引][勾引]睡醒的宝宝们都来看看这女的，没什么内涵，颜值天生有这么村姑，我也是醉了学两天就敢号称给顾客打针[打针]也太可怕了[惊恐]请问大家她自己整成这样出来吓人～真希望中国的卫生局多多处罚这样的人，太可怕了[妖怪]转发转发尤其是郑州的朋友们，大家都要注意这个女啦[惊恐][惊恐][惊恐]
  EN: Come see [flirt][flirt], wake-up babies! Check out this girl—she has no substance at all, and her looks are so rustic, it's like she was born that way [smirk]. I'm really shocked that she dares to claim she can give injections after just two days of training [injection]—that's too scary [scared]. Can everyone agree that she should have transformed herself before scaring people? I truly hope China's health bureau will punish such people more often, they're too scary [monster]. Forward this mes

From looking at the translations, it looks like there are no translation notes and the emojis are preserved!

## 5. Run on test posts

In [ ]:
results = []# one dict per post
new_count = 0 # how many posts we translated

# test_posts = data[data["split"] == "test"] #what I'm translating, change based on the split
test_posts = data[data["split"] == "train"]
# test_posts = data[data["split"] == "eval"]

# output_path = "/content/drive/MyDrive/ISchool/MIDS/266/qwen_test_baseline.csv" #change based on the split
output_path = "/content/drive/MyDrive/ISchool/MIDS/266/qwen_train_baseline.csv" #change based on the split
#output_path = "/content/drive/MyDrive/ISchool/MIDS/266/qwen_eval_baseline.csv" #change based on the split

start = time.time()

for index, row in test_posts.iterrows(): #for each test post, translate and add to results
    translation = translate(row["text"])
    new_count = new_count + 1

    results.append({
        "post_id": row["post_id"],
        "split": row["split"],
        "label": row["label"],
        "has_slang_simple": row["has_slang_simple"],
        "in_hadqaet": row["in_hadqaet"],
        "condition": "standard",
        "text": row["text"],
        "translation": translation,
    })

    # Save a backup to Drive every 100 posts, in case Colab disconnects
    if new_count % 100 == 0:
      trans = pd.DataFrame(results)
      trans.to_csv(output_path, index=False, encoding="utf-8-sig")
      print(new_count, "posts done, backup saved")

print(f"Translated {new_count} posts in {round((time.time() - start) / 60)} minutes")

100 posts done, backup saved
200 posts done, backup saved
300 posts done, backup saved
400 posts done, backup saved
500 posts done, backup saved
600 posts done, backup saved
700 posts done, backup saved
800 posts done, backup saved
900 posts done, backup saved
1000 posts done, backup saved
1100 posts done, backup saved
1200 posts done, backup saved
1300 posts done, backup saved
1400 posts done, backup saved
1500 posts done, backup saved
1600 posts done, backup saved
1700 posts done, backup saved
1800 posts done, backup saved
1900 posts done, backup saved
2000 posts done, backup saved
2100 posts done, backup saved
2200 posts done, backup saved
2300 posts done, backup saved
2400 posts done, backup saved
2500 posts done, backup saved
2600 posts done, backup saved
2700 posts done, backup saved
2800 posts done, backup saved
2900 posts done, backup saved
3000 posts done, backup saved
3100 posts done, backup saved
3200 posts done, backup saved
3300 posts done, backup saved
3400 posts done, ba

In [29]:
# Save
# Turn the list into table and save it
translations = pd.DataFrame(results)
translations.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved", len(translations), "translations to", output_path)
print()

Saved 3457 translations to /content/drive/MyDrive/ISchool/MIDS/266/qwen_eval_baseline.csv

